In [1]:
from DataExtractor import DataExtractor,DatasetBuilder
from Vocab import CharTokenizer

In [2]:
extractor = DataExtractor()
builder = DatasetBuilder(chunk_size=256)

In [3]:
mahabharat = extractor.get_from_pdf(r'pdfs/mahabharat.pdf')
builder.add_text(mahabharat,"EPIC")

In [4]:
ramayan = extractor.get_from_pdf(r'pdfs/ramayan.pdf')
builder.add_text(ramayan,"EPIC")

In [5]:
science = extractor.get_from_pdf(r'pdfs/science.pdf')
builder.add_text(science,"SCIENCE")

In [6]:
htmlcss = extractor.get_from_pdf(r'pdfs/html-css.pdf')
builder.add_text(htmlcss,"HTMLCSS")

In [7]:
java = extractor.get_from_pdf(r'pdfs/java.pdf')
builder.add_text(java,"JAVA")

In [8]:
javascript = extractor.get_from_pdf(r'pdfs/javascript.pdf')
builder.add_text(javascript,"JAVASCRIPT")

In [9]:
python = extractor.get_from_pdf(r'pdfs/python.pdf')
builder.add_text(python,"PYTHON")

In [10]:
builder.shuffle()
train,val = builder.split()
builder.save(train,r"txts/train.txt")
builder.save(val,r"txts/val.txt")

In [11]:
tokenizer = CharTokenizer()

full_text = "".join(train) + "".join(val)
tokenizer.build_vocab(full_text)
tokenizer.save_vocab(r"txts/vocab.txt")

In [14]:
tokenizer =CharTokenizer()
tokenizer.load_vocab((r"txts/vocab.txt"))
tokenizer.getvocabsize()

96

In [15]:
h = tokenizer.encode("hello")
print(tokenizer.decode(h))

hello


<!-- Dataset downloading process -->

In [6]:
%pip install datasets tqdm
%pip install zstandard


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
from datasets import load_dataset
from tqdm import tqdm
import re
import unicodedata

emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "]+",
    flags=re.UNICODE
)

def is_mostly_english(text, threshold=0.95):
    ascii_chars = sum(c.isascii() for c in text)
    return ascii_chars / max(len(text), 1) > threshold

def clean_text(text):
    text = unicodedata.normalize("NFKC", text)

    text = "".join(
        ch for ch in text
        if unicodedata.category(ch)[0] != "C"
    )

    text = emoji_pattern.sub("", text)
    text = "".join(ch for ch in text if ord(ch) < 128)

    text = re.sub(r'\s+', ' ', text)

    return text.strip()


dataset = load_dataset(
    "monology/pile-uncopyrighted",
    split="train",
    streaming=True
)

target_size = 500 * 1024 * 1024
current_size = 0

buffer = []
buffer_limit = 1000

with open("pile_500mb.txt", "w", encoding="utf-8") as f:
    for sample in tqdm(dataset):

        raw_text = sample["text"]
        text = clean_text(raw_text)

        if len(text) < 200:
            continue

        if not is_mostly_english(text):
            continue

        formatted = f"<WEB>\n{text}\n<END>\n"

        buffer.append(formatted)
        current_size += len(formatted.encode("utf-8"))

        if len(buffer) >= buffer_limit:
            f.write("".join(buffer))
            buffer = []

        if current_size >= target_size:
            break

    if buffer:
        f.write("".join(buffer))

print("Done. Clean English-only 500MB dataset created.")

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

99892it [03:20, 497.78it/s] 

Done. Clean English-only 500MB dataset created.


In [5]:
import random
import os

def load_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read().split("<END>\n")

In [18]:
old_train = load_file("C:/Users/Rohan/OneDrive/Desktop/Nano-GPT/DataHandler/txts/train.txt")
old_val = load_file("C:/Users/Rohan/OneDrive/Desktop/Nano-GPT/DataHandler/txts/val.txt")
new_pile = load_file("pile_500mb.txt")
print(old_train[1:2])
print(old_val[1:10])
old_train = [x.strip() for x in old_train if len(x.strip()) > 50]
old_val = [x.strip() for x in old_val if len(x.strip()) > 50]
new_pile = [x.strip() for x in new_pile if len(x.strip()) > 50]
print(old_train[1:2])
print(old_val[1:10])
print(new_pile[1:10])
all_samples = old_train*2 + old_val*2 + new_pile
print(all_samples[10:100])

['\n<EPIC>\nHAVING BOWED down to Narayana, and Nara the most exalted of male beings, and also to the goddess Saraswati, must the word Jaya be uttered."Vaisampayana said, \'Then those valiant descendants of Kuru, who belonged to the same party (with Virata), having joyfully celebrated the nuptials of Abhimanyu and rested themselves that night, presented themselves at dawn, well pleased, in the court of Virata, And the chamber of the king of the Matsya was full of riches, and variegated with choice gems and precious stones, with seats methodically arranged, adorned with garlands, and filled with fragrance.\n']
['\n<EPIC>\nFrom folly thou art piercing, with a piece of wood, the black cobra of virulent poison excited to fury within its hole, in desiring to fight with file:///C|/a/mahabharata/m08/m08039.htm (1 of 2)7/1/2006 9:28:17 AMDownloaded from: www.holybooks.com http://www.holybooks.com/the-mahabharata-of-vyasa-english-prose-translation/\n\nThe Mahabharata, Book 8: Karna Parva: Sectio

In [22]:
random.seed(42)
random.shuffle(all_samples)
split_ratio = 0.9
split_index = int(len(all_samples) * split_ratio)

train_samples = all_samples[:split_index]
val_samples = all_samples[split_index:]
print(train_samples[10:100])
print(val_samples[10:100])

['<WEB>\nFinite element modelling of a rotating piezoelectric ultrasonic motor.The evaluation of the performance of ultrasonic motors as a function of input parameters such as the driving frequency, voltage input and pre-load on the rotor is of key importance to their development and is here addressed by means of a finite element three-dimensional model. First the stator is simulated as a fully deformable elastic body and the travelling wave dynamics is accurately reproduced; secondly the interaction through contact between the stator and the rotor is accounted for by assuming that the rotor behaves as a rigid surface. Numerical results for the whole motor are finally compared to available experimental data.', '<SCIENCE>\nEven at this day, if the MalayArchipelago were converted into land, the tropical parts of the Indian Ocean would form a largeand perfectly enclosed basin, in which any great group of marine animals might be multiplied; andhere they would remain confined, until some of

In [9]:
def save_samples(samples, path):
    with open(path, "w", encoding="utf-8") as f:
        for sample in samples:
            f.write(sample.strip() + "\n<END>\n")

In [ ]:
os.makedirs("merged", exist_ok=True)

save_samples(train_samples, "merged/train.txt")
save_samples(val_samples, "merged/val.txt")

print("Merged and split successfully.")

Merged and split successfully.


In [2]:
def is_strict_english(text):
    if not text.isascii():
        return False
    
    letters = sum(c.isalpha() for c in text)
    if letters < 20:
        return False

    return True

In [3]:
from datasets import load_dataset
from tqdm import tqdm
import re
import unicodedata


emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "]+",
    flags=re.UNICODE
)

def is_mostly_english(text, threshold=0.95):
    ascii_chars = sum(c.isascii() for c in text)
    return ascii_chars / max(len(text), 1) > threshold

def clean_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = "".join(ch for ch in text if unicodedata.category(ch)[0] != "C")
    text = emoji_pattern.sub("", text)
    text = "".join(ch for ch in text if ord(ch) < 128)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


dataset = load_dataset("OpenAssistant/oasst1", split="train")
dataset = list(dataset) 
id_map = {sample["message_id"]: sample for sample in dataset} # type: ignore

target_size = 300 * 1024 * 1024
current_size = 0

buffer = []
buffer_limit = 500

with open("chat_300mb.txt", "w", encoding="utf-8") as f:
    for sample in tqdm(dataset):

        if sample["role"] != "assistant": # type: ignore
            continue

        parent_id = sample["parent_id"] # type: ignore
        if parent_id is None:
            continue

        parent = id_map.get(parent_id)

        if parent is None or parent["role"] != "prompter": # type: ignore
            continue

        user_text = clean_text(parent["text"]) # type: ignore
        ai_text = clean_text(sample["text"]) # type: ignore

        if len(user_text) < 10 or len(ai_text) < 20:
            continue

        if not is_strict_english(user_text + ai_text):
            continue

        formatted = (
            f"<USER>\n{user_text}\n"
            f"<AI>\n{ai_text}\n"
            f"<END>\n"
        )

        buffer.append(formatted)
        current_size += len(formatted.encode("utf-8"))

        if len(buffer) >= buffer_limit:
            f.write("".join(buffer))
            buffer = []

        if current_size >= target_size:
            break

    if buffer:
        f.write("".join(buffer))

print("Done. Chat dataset created.")

100%|██████████| 84437/84437 [00:08<00:00, 10265.30it/s]

Done. Chat dataset created.


In [ ]:
chats = load_file("chat_300mb.txt")

['<USER>\nDame una explicacin detallada de cada una de las etapas y las edades donde suelen desarrollarse.\n<AI>\nClaro!, aqu tienes: segn Piaget la etapa sensoriomotora, se extiende desde el nacimiento hasta los 24 meses de edad, y se caracteriza por ser un perodo en el que las capacidades cognitivas del infante se van desarrollando muy rpidamente. El nio o nia va adquiriendo una mayor comprensin del mundo a travs del ensayo y el error, por medio de sus sentidos y sus acciones. A principios de la etapa los bebs se caracterizan por mostrar egocentrismo extremo, es decir, no tienen ninguna comprensin del mundo aparte de su propio punto de vista actual.La etapa preoperacional se da entre los 2 y 6 aos, las capacidades cognitivas han ido desarrollndose hasta tal punto que el nio posee capacidades como hacer uso de imgenes internas, manejar esquemas, tener lenguaje y usar smbolos, los cuales sern fundamentales en el desarrollo de la conciencia propia.El principal hito de esta etapa es brin

In [11]:
random.seed(42)
random.shuffle(chats)
split_ratio = 0.9
split_index = int(len(chats) * split_ratio)

train_chats = chats[:split_index]
val_chats = chats[split_index:]
print(train_chats[10:100])
print(val_chats[10:100])

["<USER>\nWhat is openais new fund called?\n<AI>\nOpenAI's new startup fund is called Converge, and is a $100 million dollar fund for AI startups.\n", '<USER>\nqu cosa eres?\n<AI>\nSoy Open Assistant, un proyecto con el objetivo de brindar a todos acceso a un modelo de lenguaje de gran escala basado en chat. Creemos que, al hacer esto, crearemos una revolucin en la innovacin en el lenguaje. De la misma manera que la difusin estable ayud al mundo a crear arte e imgenes de nuevas maneras, esperamos que Open Assistant pueda mejorar el mundo mejorando el lenguaje en s.Como Open Assistant, estoy diseado para ser un asistente de gran capacidad para responder preguntas, generar texto y realizar tareas que requieren comprensin y generacin de lenguaje. Me alimento de una gran cantidad de datos y estoy continuamente mejorando a travs del aprendizaje automtico.Mi visin es ser el asistente del futuro, capaz de no solo escribir correos electrnicos y cartas de presentacin, sino de realizar trabajo s

In [12]:
os.makedirs("chats", exist_ok=True)
save_samples(train_chats, "chats/train.txt")
save_samples(val_chats, "chats/val.txt")

In [1]:
short_pairs = [
    ("hello", "Hello! How can I help you?"),
    ("hi", "Hi! What's up?"),
    ("how are you", "I'm doing great! How about you?"),
    ("what is your name", "I'm a small AI built using a transformer model."),
    ("who made you", "I was created as a custom AI project."),
    ("good morning", "Good morning! Hope you have a great day."),
    ("good night", "Good night! Sleep well."),
    ("thank you", "You're welcome!"),
    ("thanks", "No problem!"),
    ("what are you doing", "Just here chatting with you!"),
    ("are you real", "I'm a virtual AI model."),
    ("what is python", "Python is a popular programming language."),
    ("what is ai", "AI stands for Artificial Intelligence."),
    ("bye", "Goodbye! Talk to you later."),
    ("hey", "Hey! How can I help you?"),
    ("yo", "Yo! What's going on?"),
    ("good afternoon", "Good afternoon! How’s your day going?"),
    ("good evening", "Good evening! How can I assist you?"),
    ("nice to meet you", "Nice to meet you too!"),
    ("how is it going", "It's going well! How about you?"),
    ("what's up", "Not much! Just here to chat."),
    ("how have you been", "I've been great! Thanks for asking."),
    ("long time no see", "Yes! It's been a while."),
    ("welcome", "Thank you!"),
    ("what can you do", "I can answer questions and have conversations."),
    ("are you smart", "I try my best to be helpful!"),
    ("are you human", "No, I'm an AI model."),
    ("do you sleep", "No, I’m always available."),
    ("do you eat", "Nope! I don't need food."),
    ("do you have feelings", "I don’t have feelings, but I can understand emotions."),
    ("are you alive", "I'm not alive, but I can respond like a human."),
    ("where do you live", "I exist inside a computer system."),
    ("how do you work", "I generate responses using a transformer model."),
    ("are you free", "Yes, I'm here to help!"),
    ("what is math", "Math is the study of numbers and patterns."),
    ("what is science", "Science is the study of the natural world."),
    ("what is history", "History is the study of past events."),
    ("what is coding", "Coding is writing instructions for computers."),
    ("what is java", "Java is a programming language."),
    ("what is c++", "C++ is a powerful programming language."),
    ("what is machine learning", "Machine learning is a branch of AI."),
    ("what is the internet", "The internet connects computers worldwide."),
    ("what is a computer", "A computer processes information."),
    ("what is software", "Software is a program that runs on a computer."),
    ("i am sad", "I'm sorry to hear that. I hope things get better."),
    ("i am happy", "That's great to hear!"),
    ("i am bored", "Maybe try learning something new!"),
    ("i am tired", "Make sure to get some rest."),
    ("i feel lonely", "I'm here to chat with you."),
    ("i am excited", "That sounds exciting!"),
    ("i feel stressed", "Take a deep breath and relax."),
    ("i am nervous", "It's okay to feel nervous sometimes."),
    ("i need motivation", "You can do it! Keep going."),
    ("i feel good", "That's wonderful!"),
    ("tell me a joke", "Why did the computer go to therapy? It had too many bytes."),
    ("give me advice", "Stay consistent and keep learning."),
    ("give me a tip", "Practice makes perfect."),
    ("how to study better", "Study in short focused sessions."),
    ("how to stay healthy", "Eat well and exercise regularly."),
    ("how to learn coding", "Start with basics and practice daily."),
    ("how to improve memory", "Review information regularly."),
    ("how to focus", "Remove distractions and set small goals."),
    ("how to be confident", "Believe in yourself and practice."),
    ("how to save time", "Plan your day in advance."),
    ("tell me a joke", "Why did the computer go to therapy? It had too many bytes."),
    ("give me advice", "Stay consistent and keep learning."),
    ("give me a tip", "Practice makes perfect."),
    ("how to study better", "Study in short focused sessions."),
    ("how to stay healthy", "Eat well and exercise regularly."),
    ("how to learn coding", "Start with basics and practice daily."),
    ("how to improve memory", "Review information regularly."),
    ("how to focus", "Remove distractions and set small goals."),
    ("how to be confident", "Believe in yourself and practice."),
    ("how to save time", "Plan your day in advance."),
    ("hello", "Hi there!"),
    ("hello", "Hello!"),
    ("hello", "Hey!"),
]

In [2]:
import random
import os

def expand_pairs(pairs, multiplier=2000):
    samples = []
    for _ in range(multiplier):
        user, ai = random.choice(pairs)

        formatted = (
            f"<USER>\n{user}\n"
            f"<AI>\n{ai}\n"
            f"<END>\n"
        )

        samples.append(formatted)

    return samples

samples = expand_pairs(short_pairs, multiplier=3000)
random.shuffle(samples)
split_index = int(len(samples) * 0.9)
train_samples = samples[:split_index]
val_samples = samples[split_index:]

os.makedirs("short_chat_data", exist_ok=True)

with open("short_chat_data/train.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(train_samples))

with open("short_chat_data/val.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(val_samples))

print("Short chat dataset created and saved.")

Short chat dataset created and saved.
